# Flexible Watermark Benchmark — Google Colab
**Generation-based (Luo et al. 2026 — SDM V2.1 + WatermarkEncoder/Extractor/Decoder).**  
Runs 14 WAVES attacks, reports **BER** (bit error rate), AUROC, and TPR@1% FPR.  
No PSNR/SSIM — flexible generates images; there is no cover-vs-watermarked pair.

### Setup
1. Runtime → **GPU** (A100 for generate mode; T4 OK for pairs mode)
2. Clones [github.com/ademladhari/wavess](https://github.com/ademladhari/wavess)
3. **Pairs mode** (fast): put pre-generated pairs on Drive under `flexible/data/pairs/`
4. **Generate mode** (slow): needs `flexible/checkpoints/encoder_b*.pt` in the repo or on Drive

In [ ]:
# ── 1. Clone wavess + mount Drive ────────────────────────────────────────────
import os, sys, subprocess

REPO_URL   = 'https://github.com/ademladhari/wavess.git'
WAVES_ROOT = '/content/wavess'

if not os.path.isdir(f'{WAVES_ROOT}/flexible'):
    print('Cloning wavess (with ssl submodule)…')
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules', REPO_URL, WAVES_ROOT], check=True)
else:
    print('Repo already present:', WAVES_ROOT)

FLEX_ROOT = f'{WAVES_ROOT}/flexible'
for p in [WAVES_ROOT, FLEX_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_TO_DRIVE = True
DRIVE_OUTPUT  = '/content/drive/MyDrive/wavess_results/flexible'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

_ba = f'{WAVES_ROOT}/benchmark_attacks.py'
if not os.path.isfile(_ba):
    raise FileNotFoundError(
        f'Missing {_ba} — run: git push origin main  (from D:\\waves on your PC)'
    )
print('Repo OK:', WAVES_ROOT, '| benchmark_attacks.py found')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q diffusers transformers accelerate omegaconf einops \
               scikit-image scikit-learn tqdm datasets

In [ ]:
# ── 3. Config — edit here ────────────────────────────────────────────────────
N_IMAGES       = 100
N_BITS         = 48                  # 16, 24, or 48
DETECT_BATCH   = 16                  # extractor/decoder batch size
SEED           = 42
DEVICE         = 'cuda'
SHARED_PAYLOAD = False               # True = same watermark for all images
OUTPUT_DIR     = f'{WAVES_ROOT}/flexible/outputs_benchmark_colab'

# -- Source of watermarked images (pick ONE) --
# Option A: use pre-generated pairs (fast)
PAIRS_DIR      = f'{FLEX_ROOT}/data/pairs/b{N_BITS}/val'  # set to None if not available

# Option B: generate on the fly via SDM (slow — needs the encoder checkpoint)
# PAIRS_DIR    = None

# -- Negatives (for AUROC) --
# Folder of clean (unwatermarked) images; or None to use random noise as negatives
NEGATIVES_DIR  = None   # e.g. f'{WAVES_ROOT}/wmbench_data/negatives'
N_NEGATIVES    = N_IMAGES

In [ ]:
# ── 4. Imports ────────────────────────────────────────────────────────────────
import csv, json
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn import metrics as sk_metrics
from tqdm.auto import tqdm

sys.path.insert(0, FLEX_ROOT)
from src.data.pairs    import PairsDataset
from src.data.prompts  import load_prompts
from src.eval.extraction import bit_accuracy, tpr_at_fpr
from src.models        import WatermarkDecoder, WatermarkEncoder, WatermarkExtractor
from src.utils         import load_config

from benchmark_attacks import ATTACKS, apply_attack_tensor

device = torch.device(DEVICE)
cfg    = load_config(Path(FLEX_ROOT) / 'configs' / 'default.yaml')
print('Imports OK  |  GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# ── 5. Helpers ────────────────────────────────────────────────────────────────
_IMG_EXTS = {'.jpg','.jpeg','.png','.webp','.bmp'}

def list_images(root, n):
    paths = sorted(p for p in Path(root).iterdir()
                   if p.is_file() and p.suffix.lower() in _IMG_EXTS)
    return paths[:n]

def tensor_to_pil(t):
    arr = (t.clamp(0,1).detach().cpu().numpy() * 255).round().astype(np.uint8)
    return Image.fromarray(np.transpose(arr,(1,2,0)), 'RGB')

def pil_to_tensor(pil):
    arr = np.asarray(pil.convert('RGB'), np.float32) / 255.0
    return torch.from_numpy(np.transpose(arr,(2,0,1)))

def load_clean_tensor(path, size):
    with Image.open(path) as im:
        im = im.convert('RGB'); w,h = im.size; s = min(w,h)
        l,t = (w-s)//2,(h-s)//2
        im = im.crop((l,t,l+s,t+s))
        if im.size != (size,size): im = im.resize((size,size), Image.Resampling.LANCZOS)
        return pil_to_tensor(im)

def auroc_tpr(pos, neg, fpr=0.01):
    y = np.concatenate([np.zeros(neg.size,np.int32), np.ones(pos.size,np.int32)])
    s = np.concatenate([neg, pos])
    auc = float(sk_metrics.roc_auc_score(y, s))
    fp,tp,_ = sk_metrics.roc_curve(y, s, pos_label=1)
    idx = np.where(fp < fpr)[0]
    return auc, float(tp[idx[-1]]) if idx.size else float(tp[0])

print('Helpers defined.')

In [ ]:
# ── 6. Load extractor + decoder ───────────────────────────────────────────────
ckpt_dir = Path(FLEX_ROOT) / str(cfg.paths.checkpoints)
img_size = int(cfg.sdm.image_size)

ext = WatermarkExtractor(
    image_size=img_size,
    in_channels=3,
    latent_channels=int(cfg.sdm.latent_channels),
    latent_size=int(cfg.sdm.latent_size),
    in_downsample=int(cfg.arch.extractor.in_downsample),
    token_dim=int(cfg.arch.extractor.token_dim),
    transformer_layers=int(cfg.arch.extractor.transformer_layers),
    transformer_heads=int(cfg.arch.extractor.transformer_heads),
    mlp_hidden=int(cfg.arch.extractor.mlp_hidden),
    ffn_dim=int(cfg.arch.extractor.ffn_dim),
).to(device)
ext.load_state_dict(torch.load(ckpt_dir / f'extractor_b{N_BITS}.pt', map_location=device))
ext.eval()

dec_path = ckpt_dir / f'decoder_ft_b{N_BITS}.pt'
if not dec_path.is_file(): dec_path = ckpt_dir / f'decoder_b{N_BITS}.pt'
dec = WatermarkDecoder(
    n_bits=N_BITS,
    latent_channels=int(cfg.sdm.latent_channels),
    latent_size=int(cfg.sdm.latent_size),
    base_channels=int(cfg.arch.decoder.base_channels),
    tconv_blocks=int(cfg.arch.decoder.tconv_blocks),
    feature_hw=int(cfg.arch.encoder.latent_feature_hw),
    kernel_size=int(cfg.arch.decoder.kernel_size),
).to(device)
dec.load_state_dict(torch.load(dec_path, map_location=device))
dec.eval()

print(f'Extractor + decoder loaded (n_bits={N_BITS}, img_size={img_size})')

In [ ]:
# ── 7. Load / generate watermarked image records ──────────────────────────────
if PAIRS_DIR and Path(PAIRS_DIR).is_dir():
    print(f'Loading pre-generated pairs from {PAIRS_DIR}…')
    ds = PairsDataset(PAIRS_DIR)
    n  = min(N_IMAGES, len(ds))
    records = [{'prompt': ds[i]['prompt'], 'w': ds[i]['w'].float(),
                'image': ds[i]['image'].float()} for i in range(n)]
    print(f'Loaded {len(records)} pairs.')
else:
    print('Generating via SDM (slow)…')
    from src.models.sdm import FrozenSDM

    hf_cache = str((Path(FLEX_ROOT) / str(cfg.paths.hf_cache)).resolve())
    enc_ckpt = ckpt_dir / f'encoder_b{N_BITS}.pt'
    enc = WatermarkEncoder(
        n_bits=N_BITS,
        latent_channels=int(cfg.sdm.latent_channels),
        latent_size=int(cfg.sdm.latent_size),
        base_channels=int(cfg.arch.encoder.base_channels),
        conv_blocks=int(cfg.arch.encoder.conv_blocks),
        feature_hw=int(cfg.arch.encoder.latent_feature_hw),
        kernel_size=int(cfg.arch.encoder.kernel_size),
    ).to(device)
    enc.load_state_dict(torch.load(enc_ckpt, map_location=device))
    enc.eval()

    sdm_dtype = torch.float16 if str(cfg.sdm.dtype) == 'float16' else torch.float32
    sdm = FrozenSDM(
        pretrained_id=str(cfg.sdm.pretrained_id),
        dtype=sdm_dtype, device=device,
        enable_attention_slicing=bool(cfg.sdm.attention_slicing),
        enable_vae_slicing=bool(cfg.sdm.vae_slicing),
        enable_xformers=bool(cfg.sdm.enable_xformers),
        cache_dir=hf_cache,
    )
    prompts = load_prompts(name=str(cfg.generate_pairs.dataset_name),
                           split=str(cfg.generate_pairs.split),
                           n=N_IMAGES, seed=int(cfg.generate_pairs.seed),
                           cache_dir=hf_cache)
    gen = torch.Generator(device='cpu').manual_seed(SEED)
    records = []
    for i in tqdm(range(N_IMAGES), desc='embed/sdm'):
        w = torch.randint(0, 2, (N_BITS,), generator=gen).float()
        with torch.no_grad():
            z_img, _, _ = enc(w.unsqueeze(0).to(device))
            out = sdm.generate(z_init=z_img.to(sdm.dtype), prompt=prompts[i],
                               num_inference_steps=int(cfg.sdm.num_inference_steps),
                               guidance_scale=float(cfg.sdm.guidance_scale))
            img = out.image[0].float().cpu().clamp(0.0, 1.0)
        records.append({'prompt': prompts[i], 'w': w, 'image': img})
    del sdm, enc
    torch.cuda.empty_cache()
    print(f'Generated {len(records)} images.')

In [ ]:
# ── 8. Batched decode helper + negatives ──────────────────────────────────────
@torch.no_grad()
def decode_batch(tensor_list):
    """tensor_list: list of CHW float tensors; returns list of N-bit decoded tensors."""
    batch = torch.stack(tensor_list).to(device)
    z = ext(batch)
    wd = dec(z)
    return [wd[i].cpu() for i in range(wd.shape[0])]

# sanity check
wd0 = decode_batch([records[0]['image']])[0]
w0  = records[0]['w']
print(f'Sanity (image 0, no attack): BitAcc = {bit_accuracy(wd0.unsqueeze(0), w0.reshape(1,-1).float()):.3f}')

# negatives
neg_scores = []
ref_w = records[0]['w']

if NEGATIVES_DIR and Path(NEGATIVES_DIR).is_dir():
    neg_paths = list_images(NEGATIVES_DIR, N_NEGATIVES)
    neg_tensors = [load_clean_tensor(p, img_size) for p in neg_paths]
else:
    rng = np.random.default_rng(SEED + 1)
    neg_tensors = [torch.from_numpy(rng.random((3, img_size, img_size), dtype=np.float32))
                   for _ in range(N_NEGATIVES)]

print('Scoring negatives (batched)…')
for i in tqdm(range(0, len(neg_tensors), DETECT_BATCH), desc='negatives'):
    batch_wd = decode_batch(neg_tensors[i:i+DETECT_BATCH])
    for wd in batch_wd:
        neg_scores.append(float(bit_accuracy(wd.unsqueeze(0), ref_w.reshape(1,-1).float())))

neg_arr = np.asarray(neg_scores, np.float64)
print(f'Neg BitAcc mean: {neg_arr.mean():.4f}')

In [ ]:
# ── 9. Per-attack evaluation (batched) ────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
rows = []

for spec in ATTACKS:
    wd_list, w_list, pos_scores = [], [], []

    attacked = [apply_attack_tensor(spec.name, r['image'], seed=i)
                for i, r in enumerate(records)]

    # batched decode
    for i in range(0, len(attacked), DETECT_BATCH):
        batch_atk  = attacked[i:i+DETECT_BATCH]
        batch_recs = records[i:i+DETECT_BATCH]
        decoded    = decode_batch(batch_atk)
        for wd, r in zip(decoded, batch_recs):
            wd_list.append(wd.unsqueeze(0))
            w_list.append(r['w'].reshape(1,-1).float())
            pos_scores.append(float(bit_accuracy(wd.unsqueeze(0), r['w'].reshape(1,-1).float())))

    pos_arr = np.asarray(pos_scores, np.float64)
    auc, tpr1 = auroc_tpr(pos_arr, neg_arr)

    all_wd = torch.cat(wd_list); all_w = torch.cat(w_list)
    ba_mean = float(bit_accuracy(all_wd, all_w))
    ber = 1.0 - ba_mean  # BER = fraction of wrong bits (thesis primary metric)

    row = {'method': 'flexible', 'attack': spec.name, 'description': spec.description,
           'n_images': len(records), 'n_bits': N_BITS,
           'BER': ber, 'bit_accuracy': ba_mean,
           'AUROC': auc, 'TPR_at_1pct_FPR': tpr1}
    rows.append(row)
    print(f"  {spec.name:22s}: BER={100*ber:5.1f}%  AUROC={auc:.3f}  TPR@1%={tpr1:.3f}")

print('\nAll attacks done.')

In [ ]:
# ── 10. Save results ──────────────────────────────────────────────────────────
csv_path = Path(OUTPUT_DIR) / 'results.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
with open(Path(OUTPUT_DIR) / 'results.json', 'w') as f:
    json.dump({'method':'flexible','n_bits':N_BITS,'n_images':len(records),'attacks':rows}, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')
if SAVE_TO_DRIVE:
    import shutil
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'Copied to Drive: {DRIVE_OUTPUT}')

import pandas as pd
df = pd.DataFrame(rows)[['attack','BER','bit_accuracy','AUROC','TPR_at_1pct_FPR']]
df['BER'] = (df['BER'] * 100).round(1)
df.columns = ['Attack','BER %','BitAcc','AUROC','TPR@1%FPR']
print(df.to_string(index=False))